# WWR Segmentation — Google Colab (A100)

Semantic segmentation for building facade analysis and **Window-to-Wall Ratio (WWR)** estimation.

### Before running
1. **Runtime → Change runtime type → A100 GPU**
2. Upload `data.zip` to Google Drive: `My Drive/WWR_Seg_Model/data.zip`
3. Upload this project to Drive **or** clone from Git in cell 2

### Data flow
| Location | Purpose |
|----------|---------|
| `/content/data` | Local SSD — fast training I/O (unzipped from `data.zip`) |
| `My Drive/WWR_Seg_Model/checkpoints` | Epoch checkpoints |
| `My Drive/WWR_Seg_Model/models` | Best model |
| `My Drive/WWR_Seg_Model/logs` | CSV training history |
| `My Drive/WWR_Seg_Model/results` | Predictions, WWR, TensorBoard |

In [ ]:
# Install dependencies
!pip install -q tensorflow>=2.15 numpy pandas matplotlib scikit-learn openpyxl

## 1. Load Project Code

In [ ]:
import sys
import warnings

warnings.filterwarnings("ignore")

# ── Option A: project uploaded to Google Drive ──────────────────────────────
# Uncomment if you uploaded the repo to Drive:
# %cd /content/drive/MyDrive/WWR_Seg_Model/code

# ── Option B: clone from Git (replace with your repo URL) ───────────────────
# !git clone https://github.com/YOUR_USER/U-Net_Segmentation.git /content/WWR_Segmentation
# %cd /content/WWR_Segmentation

# ── Option C: open notebook directly from repo root ──────────────────────────
sys.path.insert(0, ".")

## 2. Colab Setup — Mount Drive & Unzip data.zip to Local Storage

In [ ]:
from WWR_Segmentation.colab_setup import setup_colab
from WWR_Segmentation.dataset import get_dataset_info

# Mount Drive → copy data.zip → unzip to /content/data (fast local I/O)
# Set force_unzip=True to re-extract after updating data.zip on Drive
config = setup_colab(force_unzip=False)

get_dataset_info(config)

## 3. Training

In [ ]:
from WWR_Segmentation.trainer import train

model, history = train(config)

## 4. Evaluation (Validation + Independent Test Set)

In [ ]:
from WWR_Segmentation.evaluate import run_evaluation

eval_results = run_evaluation(config)
print(f"Test mean IoU: {eval_results['test']['metrics']['mean_iou']['value']:.4f}")

## 5. Window-to-Wall Ratio (WWR)

In [ ]:
from WWR_Segmentation.wwr import run_wwr_analysis

wwr_df = run_wwr_analysis(config)
wwr_df.head()

## 6. Optional: 5-Fold Cross-Validation

In [ ]:
# Uncomment to run (train set only — test set is never touched)
# from WWR_Segmentation.cross_validation import run_cross_validation
# cv_summary = run_cross_validation(config)

## 7. Inference on Test Images

In [ ]:
from WWR_Segmentation.inference import run_inference

run_inference(config, input_dir=config.test_images_dir)